# Disclosure lake: data validation against the SEC sources

This notebook audits what the backfill produced against what the SEC publishes, table by table,
and ends with a scorecard. Every section states **what is checked, what the numbers mean, and
what a bad result would look like**, so the interpretation is next to the evidence.

**Where it runs.** The full audit needs the whole lake on local disk (the Mac that ran the backfill,
`LAKE_ROOT=./data`) and one of two sources of truth for the comparison:

| `VALIDATION_SOURCE` | source of truth | needs |
|---|---|---|
| `raw` | the bulk files the backfill itself downloaded (`raw/edgar/…`: `submissions.zip`, `companyfacts.zip`, `company_tickers_exchange.json`) | nothing but the lake; exact, offline, matches the backfill's own snapshot |
| `api` | `data.sec.gov` live (`submissions`, `companyfacts`, `company_tickers`) | internet and `SEC_USER_AGENT`; also shows drift since the backfill |
| `off` | none: lake-internal checks only | anything |
| `auto` (default) | `raw` when the raw files exist, else `api` when `SEC_USER_AGENT` is set, else `off` | |

On a **remote lake** (`LAKE_ROOT=s3://…`) whole-table scans are impossible at object-storage
prices, so population statistics come from the small local tables and everything per company is
computed on a **sample** (`VALIDATION_SAMPLE`, default 40 companies: the largest filers plus a random
draw). The notebook says which mode it is in and marks sampled figures as such.

```bash
# on the Mac, full audit against the backfill's own downloads:
cd H && uv sync --extra dev --extra notebooks --extra s3
LAKE_ROOT=./data SEC_USER_AGENT="Disclosure you@example.com" uv run jupyter nbconvert --to notebook --execute --inplace notebooks/data_validation.ipynb
```

In [ ]:
import os, io, json, math, random, re, sys, time, zipfile, warnings
from collections import Counter, defaultdict
from datetime import date, datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 80)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})

# nbconvert and Jupyter start the kernel inside notebooks/; work from the repository root so that
# .env is found and a relative LAKE_ROOT such as ./data means the same lake the CLI uses.
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "pyproject.toml").exists() and (_candidate / "filings_hub").is_dir():
        os.chdir(_candidate)
        break
from filings_hub.config import get_settings
LAKE_ROOT = os.environ.get("LAKE_ROOT") or get_settings().resolved_lake_root()   # .env is honoured, as the CLI does
SOURCE_MODE = os.environ.get("VALIDATION_SOURCE", "auto").lower()
SAMPLE_SIZE = int(os.environ.get("VALIDATION_SAMPLE", "40"))
FULL_SCANS = os.environ.get("VALIDATION_FULL", "").lower() in ("1", "true", "yes")
SEC_USER_AGENT = os.environ.get("SEC_USER_AGENT", "")
RANDOM_SEED = 7
TODAY = date.today()

from filings_hub.lake import layout
from filings_hub.lake.storage import Storage
from filings_hub.db.database import Database

storage = Storage(LAKE_ROOT)
REMOTE = storage.is_remote
FULL_SCANS = FULL_SCANS or not REMOTE   # a local lake is scanned whole; a remote one only on request
db = Database("", storage, lazy_remote_views=REMOTE)
if REMOTE:
    t0 = time.time(); db.warm(); print(f"filings table bound over object storage in {time.time()-t0:.0f}s")

def q(sql, params=()):
    """A query as a DataFrame (column names survive an empty result)."""
    if db.backend == "duckdb":
        return db.duck.fetch_arrow(sql, list(params)).to_pandas()
    return pd.DataFrame(db.query(sql, list(params)))

def company_rows(name, cik, where="", params=()):
    """One company's rows of a per-company table (a partition read on a remote lake)."""
    return q(f"SELECT * FROM {db.table(name, cik)} WHERE cik = ? {where}", [cik, *params])

SCORECARD = []
def score(check, value, threshold, status, note=""):
    SCORECARD.append({"check": check, "value": value, "threshold": threshold, "status": status, "note": note})
    print(f"[{status}] {check}: {value}  (threshold {threshold}) {note}")

def pct(n, d):
    return float("nan") if not d else 100.0 * n / d

print(f"lake: {storage.root}  ({'remote object storage: sampled per-company checks' if REMOTE else 'local disk: full scans'})")
if q("SELECT count(*) AS n FROM companies").n.iloc[0] == 0:
    raise SystemExit(f"The lake at {storage.root} has no companies table. Point LAKE_ROOT at the backfilled lake "
                     "(the value in .env, or s3://bucket/prefix) and run again.")
print(f"source mode requested: {SOURCE_MODE}; sample size: {SAMPLE_SIZE}; full scans: {FULL_SCANS}; today: {TODAY}")

## 0. Source of truth: what this run can compare against

`raw` is preferred: the backfill downloaded the SEC's bulk files and kept them under `raw/edgar/`, so
comparing against them checks *the loader*, with no network and no drift. `api` checks the loader
**and** shows what the SEC has published since. If neither is available the notebook still runs every
lake-internal check (structure, integrity, reconciliation between independently sourced tables).

In [ ]:
def _latest(pattern):
    hits = sorted(storage.glob(pattern))
    return hits[-1] if hits else None

RAW = {
    "submissions_zip": _latest(f"{layout.RAW}/submissions/*/submissions.zip"),
    "companyfacts_zip": _latest(f"{layout.RAW}/companyfacts/*/companyfacts.zip"),
    "company_tickers": _latest(f"{layout.RAW}/company_tickers/*/company_tickers_exchange.json"),
}
raw_available = any(RAW.values())
api_available = bool(SEC_USER_AGENT and "@" in SEC_USER_AGENT)

if SOURCE_MODE == "auto":
    SOURCE_MODE = "raw" if raw_available else ("api" if api_available else "off")
if SOURCE_MODE == "raw" and not raw_available:
    print("raw files not found in the lake: falling back to", "api" if api_available else "off"); SOURCE_MODE = "api" if api_available else "off"
if SOURCE_MODE == "api" and not api_available:
    print("SEC_USER_AGENT is not set (SEC requires 'Name email'): source checks are off"); SOURCE_MODE = "off"

edgar = None
if SOURCE_MODE == "api":
    from filings_hub.ingest.edgar_client import EdgarClient
    edgar = EdgarClient(SEC_USER_AGENT)
    try:
        edgar.get_json(EdgarClient.submissions_url(320193))
        print("EDGAR reachable: live comparison against data.sec.gov")
    except Exception as e:
        print(f"EDGAR not reachable from here ({type(e).__name__}): source checks are off"); SOURCE_MODE = "off"; edgar = None

print("source mode:", SOURCE_MODE)
for k, v in RAW.items():
    print(f"  raw {k}: {v or '-'}")

## 1. Inventory and pipeline history

**Checked:** which tables exist, their sizes, and what the backfill's own run log and the FSDS load
log recorded. **Meaning:** the run log is the pipeline's account of each stage (rows loaded, quarters
built, errors). The FSDS load log reconciles raw rows in each SEC quarterly file with the rows loaded
and rejected. **Bad looks like:** a table missing, a run that stopped before the statements stage, a
quarter with rejected rows or an unparsed-value count that is not zero.

In [ ]:
from filings_hub.ingest.fsds import load_log

tables = {
    "companies": layout.COMPANIES, "tickers": layout.TICKERS, "periods": layout.PERIODS,
    "company_metrics": layout.COMPANY_METRICS, "filings": layout.FILINGS, "facts": layout.FACTS,
    "statements": layout.STATEMENTS, "statement_checks": layout.STATEMENT_CHECKS,
    "documents": layout.DOCUMENTS, "fsds/sub": f"{layout.FSDS}/sub", "run_log": layout.RUN_LOG,
}
inv = []
for name, rel in tables.items():
    present = storage.any_parquet_under(rel) if not rel.endswith(".parquet") else storage.exists(rel)
    rows = None
    if present and rel.endswith(".parquet"):
        rows = q(f"SELECT count(*) AS n FROM {name}").n.iloc[0]
    elif present and name == "filings":
        rows = q("SELECT count(*) AS n FROM filings").n.iloc[0]  # answered from parquet footers
    inv.append({"table": name, "path": rel, "present": present, "rows": rows})
inventory = pd.DataFrame(inv)
display(inventory)
missing = inventory[~inventory.present].table.tolist()
score("all serving tables present", "none missing" if not missing else f"missing: {missing}", "none missing", "PASS" if not missing else "FAIL")

In [ ]:
runs = q("SELECT run_id, kind, started_at, finished_at, duration_seconds, status, new_filings, ciks_refreshed, facts_rows, statements_built, fsds_quarters_loaded, failures, error, steps FROM run_log ORDER BY started_at")
if len(runs):
    show = runs.copy()
    show["fsds_quarters"] = show.fsds_quarters_loaded.map(lambda v: len(v) if isinstance(v, (list, np.ndarray)) else 0)
    show["error"] = show.error.fillna("").str.slice(0, 90)
    show["steps"] = show.steps.map(lambda v: ", ".join(v) if isinstance(v, (list, np.ndarray)) else "")
    display(show[["run_id", "kind", "started_at", "duration_seconds", "status", "new_filings", "ciks_refreshed", "facts_rows", "fsds_quarters", "steps", "error"]])
    last = runs.iloc[-1]
    steps = list(last.steps) if isinstance(last.steps, (list, np.ndarray)) else []
    reached_statements = any(s.startswith("statements") for s in steps)
    fig, ax = plt.subplots()
    ax.barh(runs.run_id, runs.duration_seconds / 60, color=["#2a9d8f" if s == "ok" else "#e76f51" for s in runs.status])
    ax.set_xlabel("minutes"); ax.set_title("Pipeline runs: duration and outcome (green ok, red failed)")
    plt.show()
    print("Interpretation: the last run's step list shows how far it got. A run that reached the statements stage built the")
    print("statement tables even if a later stage (fallbacks, metrics) failed and was completed by `filings-hub finish`,")
    print("which does not write a run-log row. Errors in earlier runs are informational once a later run passed the stage.")
    score("last run reached the statements stage", reached_statements, True, "PASS" if reached_statements else "FAIL", f"last status={last.status}")
else:
    print("no run_log rows"); score("run log present", False, True, "WARN")

In [ ]:
ll = pd.DataFrame(load_log(storage))
if len(ll):
    ll["quarter_key"] = ll.quarter
    pivot = ll.pivot_table(index="quarter", columns="table", values=["raw_rows", "loaded_rows", "rejected_rows", "unparsed_values"], aggfunc="sum").sort_index()
    rej = ll.groupby("quarter")[["rejected_rows", "unparsed_values"]].sum()
    tot = ll.groupby("quarter")[["raw_rows", "loaded_rows"]].sum()
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(tot.index, tot.raw_rows / 1e6, label="raw rows in SEC file", marker=".")
    ax.plot(tot.index, tot.loaded_rows / 1e6, label="rows loaded", marker=".")
    ax.set_ylabel("millions of rows (sub+num+pre+tag)"); ax.set_title("FSDS quarterly files: raw vs loaded"); ax.legend()
    ax.set_xticks(range(0, len(tot.index), 4)); ax.set_xticklabels(tot.index[::4], rotation=45)
    plt.show()
    num = ll[ll.table == "num"].set_index("quarter").sort_index()
    print("num table: loaded/raw per quarter (the loader drops rows it cannot type, e.g. non-numeric values):")
    print((num.loaded_rows / num.raw_rows).describe().round(4))
    bad = ll[(ll.rejected_rows > 0) | (ll.unparsed_values > 0)]
    display(bad[["quarter", "table", "raw_rows", "loaded_rows", "rejected_rows", "unparsed_values", "reject_examples"]] if len(bad) else pd.DataFrame({"note": ["no quarter reported rejected or unparsed rows"]}))
    quarters = sorted(ll.quarter.unique())
    expected = []
    y, qn = 2009, 2
    while (y, qn) <= (TODAY.year, (TODAY.month - 1) // 3 + 1):
        expected.append(f"{y}q{qn}"); qn += 1
        if qn == 5: y, qn = y + 1, 1
    gaps = [x for x in expected[:-1] if x not in quarters]   # the current quarter's file is published after quarter end
    score("FSDS quarters loaded contiguously since 2009q2", f"{len(quarters)} quarters, gaps: {gaps or 'none'}", "no gaps", "PASS" if not gaps else "FAIL")
    score("FSDS rows rejected by the loader", int(ll.rejected_rows.sum()), 0, "PASS" if ll.rejected_rows.sum() == 0 else "WARN")
else:
    print("no FSDS load log"); score("FSDS load log present", False, True, "WARN")

## 2. Universe: companies and tickers

**Checked:** the companies and tickers tables against the SEC's `company_tickers_exchange.json` (every
listed ticker with its CIK and exchange), plus internal sanity: duplicates, blank names, SIC coverage.
**Meaning:** the universe is the entry point of the product (search); a ticker the SEC lists that the
lake lacks is a company no one can find. **Bad looks like:** ticker recall below ~99%, many CIKs in
the SEC list with no company row, duplicate primary tickers per company.

In [ ]:
companies = q("SELECT cik, name, ticker, exchange, sic, sic_description, entity_type, is_listed, is_active, last_filing_date, last_financial_report_date, filing_count FROM companies")
tickers = q("SELECT cik, ticker, exchange, is_primary, source FROM tickers")
print(f"companies: {len(companies):,}   listed: {int(companies.is_listed.fillna(False).sum()):,}   active: {int(companies.is_active.fillna(False).sum()):,}   with SIC: {companies.sic.notna().sum():,}")
print(f"tickers: {len(tickers):,} rows, {tickers.cik.nunique():,} companies, primary flags: {int(tickers.is_primary.fillna(False).sum()):,}")

dup_cik = companies.cik.duplicated().sum()
blank_names = companies.name.isna().sum() + (companies.name.fillna("").str.strip() == "").sum()
multi_primary = tickers[tickers.is_primary.fillna(False)].groupby("cik").size().gt(1).sum()
score("companies: duplicate CIK rows", int(dup_cik), 0, "PASS" if dup_cik == 0 else "FAIL")
score("companies: blank names", int(blank_names), 0, "PASS" if blank_names == 0 else "WARN")
score("tickers: companies with more than one primary ticker", int(multi_primary), 0, "PASS" if multi_primary == 0 else "WARN")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
tickers.exchange.fillna("(none)").value_counts().head(8).plot.bar(ax=axes[0], color="#264653"); axes[0].set_title("tickers by exchange"); axes[0].set_xlabel("")
companies.sic_description.fillna("(no SIC)").value_counts().head(15).sort_values().plot.barh(ax=axes[1], color="#2a9d8f"); axes[1].set_title("companies by industry (top 15)")
plt.tight_layout(); plt.show()

In [ ]:
from filings_hub.ingest.sync_universe import parse_company_tickers
from filings_hub.ingest.edgar_client import COMPANY_TICKERS_EXCHANGE_URL

src_tickers = None
if SOURCE_MODE == "raw" and RAW["company_tickers"]:
    src_tickers = pd.DataFrame(parse_company_tickers(storage.read_bytes(RAW["company_tickers"])))
elif SOURCE_MODE == "api":
    src_tickers = pd.DataFrame(parse_company_tickers(edgar.get(COMPANY_TICKERS_EXCHANGE_URL).content))

if src_tickers is not None and len(src_tickers):
    src_set = set(zip(src_tickers.cik, src_tickers.ticker.str.upper()))
    lake_set = set(zip(tickers.cik, tickers.ticker.str.upper()))
    missing = src_set - lake_set
    extra = lake_set - src_set
    recall = pct(len(src_set & lake_set), len(src_set))
    src_ciks = set(src_tickers.cik); lake_ciks = set(companies.cik)
    no_company = src_ciks - lake_ciks
    print(f"SEC lists {len(src_set):,} (cik, ticker) pairs; lake has {len(lake_set):,}; recall {recall:.2f}%; extra in lake {len(extra):,}")
    print(f"SEC CIKs with no company row in the lake: {len(no_company):,}")
    if missing:
        display(src_tickers[src_tickers.apply(lambda r: (r.cik, r.ticker.upper()) in missing, axis=1)].head(20))
    print("Interpretation: extras are usually tickers the SEC has since dropped (delistings) or secondary listings the")
    print("backfill kept; misses are new listings since the snapshot. Both grow with the age of the snapshot.")
    score("ticker recall vs SEC company_tickers", f"{recall:.2f}%", ">= 99%", "PASS" if recall >= 99 else ("WARN" if recall >= 95 else "FAIL"))
    score("SEC-listed CIKs without a company row", len(no_company), "<= 0.5% of list", "PASS" if pct(len(no_company), len(src_ciks)) <= 0.5 else "WARN")
else:
    print("no ticker source in this mode: skipped")

## 3. Filings

**Checked:** volume per year and per form, date sanity, duplicates, XBRL share; then for a sample of
companies the lake's filing list against the SEC's `submissions` record for that company (the
authoritative list of everything the company ever filed). **Meaning:** the filings table is the spine
every other table hangs on (periods point at accessions, statements are per accession). Recall against
`submissions` tells whether the loader kept every filing; a form or date mismatch on a shared accession
tells whether it kept them correctly. **Bad looks like:** recall below ~99.5% for filings before the
snapshot date, duplicated accessions, years with implausibly few filings.

In [ ]:
by_year = q("SELECT year, count(*) AS filings, count(DISTINCT cik) AS companies FROM filings GROUP BY year ORDER BY year")
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(by_year.year, by_year.filings / 1e6, color="#264653"); ax.set_ylabel("millions of filings"); ax.set_title("Filings per year in the lake")
ax2 = ax.twinx(); ax2.plot(by_year.year, by_year.companies / 1e3, color="#e76f51", marker="."); ax2.set_ylabel("thousand filers (red)")
plt.show()
print(f"total filings: {by_year.filings.sum():,}  years: {by_year.year.min()}-{by_year.year.max()}")
ref = by_year[(by_year.year >= 2004) & (by_year.year < TODAY.year)]
thin = ref[ref.filings < 0.5 * ref.filings.median()]
score("years since 2004 with under half the median filing volume", thin.year.tolist() or "none", "none", "PASS" if len(thin) == 0 else "WARN", "EDGAR volume only stabilised around 2003 (electronic filing became mandatory in stages)")

In [ ]:
if FULL_SCANS:
    forms = q("SELECT form, count(*) AS n FROM filings GROUP BY form ORDER BY n DESC")
    dups = q("SELECT count(*) - count(DISTINCT cik || '|' || accession) AS dup FROM filings").dup.iloc[0]
    joint = q("SELECT count(*) - count(DISTINCT accession) AS n FROM filings").n.iloc[0]
    nulls = q("SELECT sum(filed_date IS NULL) AS no_date, sum(form IS NULL OR form = '') AS no_form, sum(primary_doc IS NULL) AS no_primary FROM filings").iloc[0]
    xbrl = q("SELECT year, avg(CASE WHEN is_xbrl THEN 1 ELSE 0 END) AS xbrl_share, avg(CASE WHEN is_inline_xbrl THEN 1 ELSE 0 END) AS ixbrl_share FROM filings WHERE year >= 2005 GROUP BY year ORDER BY year")
    scope_note = "whole table"
else:
    forms = q(f"SELECT form, count(*) AS n FROM filings WHERE year >= {TODAY.year - 1} GROUP BY form ORDER BY n DESC")
    dups = q(f"SELECT count(*) - count(DISTINCT cik || '|' || accession) AS dup FROM filings WHERE year >= {TODAY.year - 1}").dup.iloc[0]
    joint = q(f"SELECT count(*) - count(DISTINCT accession) AS n FROM filings WHERE year >= {TODAY.year - 1}").n.iloc[0]
    nulls = q(f"SELECT sum(filed_date IS NULL) AS no_date, sum(form IS NULL OR form = '') AS no_form, sum(primary_doc IS NULL) AS no_primary FROM filings WHERE year >= {TODAY.year - 1}").iloc[0]
    xbrl = q(f"SELECT year, avg(CASE WHEN is_xbrl THEN 1 ELSE 0 END) AS xbrl_share, avg(CASE WHEN is_inline_xbrl THEN 1 ELSE 0 END) AS ixbrl_share FROM filings WHERE year >= {TODAY.year - 1} GROUP BY year ORDER BY year")
    scope_note = f"last two years only (remote lake; set VALIDATION_FULL=1 to scan everything)"
print("scope:", scope_note)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
forms.head(20).set_index("form").n.plot.bar(ax=axes[0], color="#2a9d8f"); axes[0].set_title("filings by form (top 20)"); axes[0].set_xlabel("")
axes[1].plot(xbrl.year, 100 * xbrl.xbrl_share, marker=".", label="XBRL"); axes[1].plot(xbrl.year, 100 * xbrl.ixbrl_share, marker=".", label="inline XBRL"); axes[1].set_title("share of filings with XBRL data (%)"); axes[1].legend()
plt.tight_layout(); plt.show()
print(f"duplicate (cik, accession) rows: {dups:,}   accessions shared by several filers (joint filings, e.g. a Form 4 with two reporting persons): {joint:,}   null filed_date: {int(nulls.no_date):,}   blank form: {int(nulls.no_form):,}   no primary document: {int(nulls.no_primary):,}")
score("filings: duplicate (cik, accession) rows", int(dups), 0, "PASS" if dups == 0 else "FAIL", scope_note + "; an accession appears once per filer, so joint filings are expected to repeat it")
score("filings: rows without a filed date", int(nulls.no_date), 0, "PASS" if nulls.no_date == 0 else "FAIL", scope_note)

### 3.1 Choosing the sample of companies

The same sample serves every per-company comparison below: the largest filers (they exercise every
code path: many forms, amendments, restatements) plus a random draw among companies that have
statements and metrics (the product's serving set). The random half is what makes the recall numbers
an estimate for the population rather than for famous companies only.

In [ ]:
metrics_ciks = set(q("SELECT DISTINCT cik FROM company_metrics").cik)
served = companies[companies.cik.isin(metrics_ciks)]
random.seed(RANDOM_SEED)
top_n = max(5, SAMPLE_SIZE // 2)
top = served.sort_values("filing_count", ascending=False).head(top_n).cik.tolist()
rest = [c for c in served.cik.tolist() if c not in top]
rand = random.sample(rest, min(SAMPLE_SIZE - len(top), len(rest)))
SAMPLE = top + rand
names = companies.set_index("cik").name
print(f"{len(SAMPLE)} companies: {len(top)} largest filers + {len(rand)} random among {len(served):,} companies with metrics")
display(companies[companies.cik.isin(SAMPLE)][["cik", "name", "ticker", "sic_description", "filing_count"]].sort_values("filing_count", ascending=False).head(12))

In [ ]:
from filings_hub.ingest.submissions import parse_filings, merge_pages, CIK_FILE_RE, PAGE_FILE_RE
from filings_hub.ingest.edgar_client import EdgarClient

def source_submissions(cik):
    """The SEC's full filing list for one company in the current source mode, or None."""
    if SOURCE_MODE == "raw" and RAW["submissions_zip"]:
        with storage.local_copy(RAW["submissions_zip"]) as path:
            with zipfile.ZipFile(path) as zf:
                name = f"CIK{cik:010d}.json"
                if name not in zf.namelist():
                    return None
                data = json.loads(zf.read(name))
                pages = sorted((int(m.group(2)), n) for n in zf.namelist() if (m := PAGE_FILE_RE.match(n)) and int(m.group(1)) == cik)
                data = merge_pages(data, [json.loads(zf.read(n)) for _, n in pages])
        return parse_filings(data, "raw")
    if SOURCE_MODE == "api":
        data = edgar.get_json(EdgarClient.submissions_url(cik))
        pages = [edgar.get_json(EdgarClient.submissions_page_url(f["name"])) for f in (data.get("filings", {}).get("files") or [])]
        return parse_filings(merge_pages(data, pages), "api")
    return None

filing_cmp = []
t0 = time.time()
if SOURCE_MODE != "off":
    for cik in SAMPLE:
        src = source_submissions(cik)
        if src is None:
            continue
        s = pd.DataFrame(src)
        l = q("SELECT accession, form, filed_date FROM filings WHERE cik = ?", [cik])
        src_acc, lake_acc = set(s.accession), set(l.accession)
        both = src_acc & lake_acc
        merged = s.set_index("accession").loc[list(both)][["form", "filed_date"]].join(l.set_index("accession")[["form", "filed_date"]], rsuffix="_lake")
        form_mismatch = int((merged.form.fillna("") != merged.form_lake.fillna("")).sum())
        date_mismatch = int((pd.to_datetime(merged.filed_date) != pd.to_datetime(merged.filed_date_lake)).sum())
        snapshot = pd.to_datetime(l.filed_date).max()
        missing_before_snapshot = [a for a in src_acc - lake_acc if pd.to_datetime(s.set_index('accession').filed_date[a]) <= snapshot]
        filing_cmp.append({"cik": cik, "name": names.get(cik, "")[:32], "source": len(src_acc), "lake": len(lake_acc), "shared": len(both),
                           "missing": len(src_acc - lake_acc), "missing_before_lake_max_date": len(missing_before_snapshot), "extra_in_lake": len(lake_acc - src_acc),
                           "form_mismatch": form_mismatch, "date_mismatch": date_mismatch, "recall_pct": pct(len(both), len(src_acc))})
    print(f"compared {len(filing_cmp)} companies in {time.time()-t0:.0f}s")
if filing_cmp:
    fc = pd.DataFrame(filing_cmp).sort_values("recall_pct")
    display(fc)
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(range(len(fc)), fc.recall_pct, color=["#2a9d8f" if r >= 99.5 else "#e9c46a" if r >= 98 else "#e76f51" for r in fc.recall_pct])
    ax.set_ylim(min(90, fc.recall_pct.min() - 1), 100.5); ax.set_xticks(range(len(fc))); ax.set_xticklabels(fc.name, rotation=90, fontsize=7); ax.set_ylabel("% of SEC filings present"); ax.set_title("Filing recall per sampled company vs SEC submissions")
    plt.tight_layout(); plt.show()
    overall = pct(fc.shared.sum(), fc.source.sum())
    strict = pct(fc.shared.sum(), fc.shared.sum() + fc.missing_before_lake_max_date.sum())
    print("Interpretation: 'missing' counts everything the SEC lists that the lake lacks; 'missing_before_lake_max_date' removes")
    print("filings newer than the lake's latest filing (those are drift since the backfill, not loader loss).")
    print(f"recall overall {overall:.2f}%; recall excluding post-snapshot filings {strict:.2f}%")
    score("filing recall vs SEC submissions (sample, pre-snapshot)", f"{strict:.2f}%", ">= 99.5%", "PASS" if strict >= 99.5 else ("WARN" if strict >= 98 else "FAIL"))
    score("form mismatches on shared accessions", int(fc.form_mismatch.sum()), 0, "PASS" if fc.form_mismatch.sum() == 0 else "WARN")
    score("filed-date mismatches on shared accessions", int(fc.date_mismatch.sum()), 0, "PASS" if fc.date_mismatch.sum() == 0 else "WARN")
else:
    print("source comparison of filings skipped in this mode")

## 4. XBRL facts

**Checked:** the facts table (every numeric XBRL fact the SEC exposes per company through
`companyfacts`) for shape and, per sampled company, against the SEC's `companyfacts` document: are
the same (concept, unit, period, accession) keys present, and do the values agree? **Meaning:** facts
are the second, independent source of the financial numbers (statements come from the quarterly
FSDS files); section 7 relies on them. **Bad looks like:** key recall below ~99%, any value
disagreement on shared keys (the loader copies values; a mismatch is a parsing bug), taxonomies
missing (`us-gaap`, `dei`, `ifrs-full`).

In [ ]:
fact_frames = []
for cik in SAMPLE:
    f = company_rows("facts", cik)
    if len(f):
        f["_cik"] = cik; fact_frames.append(f)
facts_s = pd.concat(fact_frames) if fact_frames else pd.DataFrame()
print(f"sampled facts rows: {len(facts_s):,} over {facts_s._cik.nunique() if len(facts_s) else 0} companies")
if len(facts_s):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    facts_s.taxonomy.value_counts().plot.bar(ax=axes[0], color="#264653"); axes[0].set_title("facts by taxonomy (sample)"); axes[0].set_xlabel("")
    facts_s.duration_kind.fillna("(none)").value_counts().plot.bar(ax=axes[1], color="#2a9d8f"); axes[1].set_title("duration kind"); axes[1].set_xlabel("")
    per_company = facts_s.groupby("_cik").size(); axes[2].hist(per_company, bins=20, color="#e9c46a"); axes[2].set_title("facts per sampled company"); axes[2].set_xlabel("rows")
    plt.tight_layout(); plt.show()
    print("top concepts:"); print(facts_s.concept.value_counts().head(12).to_string())
    null_val = int(facts_s.value.isna().sum()); null_end = int(facts_s.period_end.isna().sum())
    score("facts: null values / null period_end (sample)", f"{null_val} / {null_end}", "0 / 0", "PASS" if null_val == 0 and null_end == 0 else "FAIL")
    with_gaap = facts_s[facts_s.taxonomy.isin(["us-gaap", "ifrs-full"])]._cik.nunique()
    score("facts: us-gaap or ifrs-full present for sampled companies", f"{with_gaap} of {facts_s._cik.nunique()}", "all", "PASS" if with_gaap == facts_s._cik.nunique() else "WARN", "a company with only dei facts filed no financial statements in XBRL")
if FULL_SCANS:
    tot = q("SELECT count(*) AS n, count(DISTINCT cik) AS companies FROM facts").iloc[0]
    print(f"whole table: {int(tot.n):,} facts for {int(tot.companies):,} companies")

In [ ]:
from filings_hub.ingest.sync_facts import parse_companyfacts

def source_companyfacts(cik):
    if SOURCE_MODE == "raw" and RAW["companyfacts_zip"]:
        with storage.local_copy(RAW["companyfacts_zip"]) as path:
            with zipfile.ZipFile(path) as zf:
                name = f"CIK{cik:010d}.json"
                if name not in zf.namelist():
                    return None
                return parse_companyfacts(json.loads(zf.read(name)))
    if SOURCE_MODE == "api":
        try:
            return parse_companyfacts(edgar.get_json(EdgarClient.companyfacts_url(cik)))
        except Exception as e:
            print(f"  CIK {cik}: {type(e).__name__}"); return None
    return None

KEY = ["taxonomy", "concept", "unit", "period_start", "period_end", "accession"]
fact_cmp, value_diffs = [], []
if SOURCE_MODE != "off" and len(facts_s):
    for cik in SAMPLE:
        src = source_companyfacts(cik)
        if not src:
            continue
        s = pd.DataFrame(src); l = facts_s[facts_s._cik == cik]
        if l.empty:
            continue
        for d in (s, l):
            d["period_start"] = pd.to_datetime(d.period_start).dt.date; d["period_end"] = pd.to_datetime(d.period_end).dt.date
        s = s.drop_duplicates(KEY); l = l.drop_duplicates(KEY)
        m = s.merge(l, on=KEY, how="outer", suffixes=("_src", "_lake"), indicator=True)
        both = m[m._merge == "both"]
        rel = ((both.value_src - both.value_lake).abs() / both.value_src.abs().replace(0, np.nan)).fillna((both.value_src - both.value_lake).abs())
        value_diffs.extend(rel.tolist())
        fact_cmp.append({"cik": cik, "name": names.get(cik, "")[:32], "source_keys": int((m._merge != "right_only").sum()), "lake_keys": int((m._merge != "left_only").sum()),
                         "shared": len(both), "missing": int((m._merge == "left_only").sum()), "extra_in_lake": int((m._merge == "right_only").sum()),
                         "value_mismatch": int((rel > 1e-9).sum()), "recall_pct": pct(len(both), int((m._merge != "right_only").sum()))})
if fact_cmp:
    fcm = pd.DataFrame(fact_cmp).sort_values("recall_pct"); display(fcm)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].bar(range(len(fcm)), fcm.recall_pct, color="#2a9d8f"); axes[0].set_xticks(range(len(fcm))); axes[0].set_xticklabels(fcm.name, rotation=90, fontsize=7); axes[0].set_ylim(min(90, fcm.recall_pct.min() - 1), 100.5); axes[0].set_title("fact key recall vs SEC companyfacts")
    vd = np.array(value_diffs); axes[1].hist(np.log10(vd[vd > 0] + 1e-15), bins=30, color="#e76f51"); axes[1].set_title(f"log10 relative value difference on shared keys ({int((vd > 1e-9).sum())} non-zero of {len(vd):,})")
    plt.tight_layout(); plt.show()
    recall = pct(fcm.shared.sum(), fcm.source_keys.sum())
    print("Interpretation: the lake de-duplicates restated facts and keeps the latest value per key, so 'extra' should be ~0 and")
    print("'missing' small; a value mismatch means the loader changed a number and must be investigated.")
    score("fact key recall vs SEC companyfacts (sample)", f"{recall:.2f}%", ">= 99%", "PASS" if recall >= 99 else ("WARN" if recall >= 97 else "FAIL"))
    score("fact value mismatches on shared keys", int(fcm.value_mismatch.sum()), 0, "PASS" if fcm.value_mismatch.sum() == 0 else "FAIL")
else:
    print("source comparison of facts skipped in this mode")

## 5. Statements from the SEC financial statement data sets (FSDS)

**Checked:** how many filings got statements per quarter compared with the filings in the SEC's
quarterly `sub` file, which statement types were built, the share coming from fallbacks (built from
facts when FSDS had no presentation for a filing), and the arithmetic checks the builder ran (assets =
liabilities + equity, subtotals add up, cash flow reconciles). **Meaning:** this is the table the
product shows; the checks are its internal evidence of correctness. **Bad looks like:** a quarter
where far fewer statements than `sub` filings were built, a check pass rate well under ~95%, whole
statement types missing for large filers.

In [ ]:
stmt_frames, check_frames = [], []
for cik in SAMPLE:
    s = company_rows("statements", cik); c = company_rows("statement_checks", cik)
    if len(s): s["_cik"] = cik; stmt_frames.append(s)
    if len(c): c["_cik"] = cik; check_frames.append(c)
stmts = pd.concat(stmt_frames) if stmt_frames else pd.DataFrame()
checks = pd.concat(check_frames) if check_frames else pd.DataFrame()
print(f"sampled statements rows: {len(stmts):,} for {stmts._cik.nunique() if len(stmts) else 0} companies; check rows: {len(checks):,}")
if len(stmts):
    per_acc = stmts.groupby(["_cik", "accession"]).statement.agg(lambda x: "".join(sorted(set(x) & {"IS", "BS", "CF"}))).reset_index()
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    per_acc.statement.replace("", "(none of IS/BS/CF)").value_counts().plot.bar(ax=axes[0], color="#264653"); axes[0].set_title("statement types per filing (sample)"); axes[0].set_xlabel("")
    stmts.groupby("source").accession.nunique().plot.bar(ax=axes[1], color="#2a9d8f"); axes[1].set_title("filings by statement source"); axes[1].set_xlabel("")
    yr = stmts.assign(year=pd.to_datetime(stmts.filed_date).dt.year).groupby("year").accession.nunique(); axes[2].plot(yr.index, yr.values, marker="."); axes[2].set_title("filings with statements per year (sample)")
    plt.tight_layout(); plt.show()
    complete = pct((per_acc.statement == "BCFIS").sum() + (per_acc.statement == "BSCFIS").sum(), len(per_acc))
    full3 = pct(per_acc.statement.map(lambda s: {"IS","BS","CF"} <= set(re.findall("IS|BS|CF", s))).sum(), len(per_acc))
    fallback_share = pct(stmts[stmts.source != "fsds"].accession.nunique(), stmts.accession.nunique())
    print(f"filings with all three primary statements: {full3:.1f}%   fallback share of filings: {fallback_share:.1f}%")
    score("filings with IS, BS and CF (sample)", f"{full3:.1f}%", ">= 85%", "PASS" if full3 >= 85 else "WARN", "10-Q/10-K only carry all three; other forms legitimately lack some")
    dup_lines = int(stmts.duplicated(["accession", "statement", "report", "line", "period_start", "period_end", "is_parenthetical"]).sum())
    score("statements: duplicate presentation lines (sample)", dup_lines, 0, "PASS" if dup_lines == 0 else "WARN")
if len(checks):
    rate = checks.groupby(["statement", "check_name"]).passed.agg(["mean", "size"]).reset_index(); rate["pass_pct"] = 100 * rate["mean"]
    display(rate[["statement", "check_name", "size", "pass_pct"]].sort_values("pass_pct"))
    overall = 100 * checks.passed.mean()
    fig, ax = plt.subplots(figsize=(10, 4)); by_year = checks.merge(stmts[["accession", "filed_date"]].drop_duplicates("accession"), on="accession", how="left")
    by_year["year"] = pd.to_datetime(by_year.filed_date).dt.year; g = by_year.groupby("year").passed.mean() * 100; ax.plot(g.index, g.values, marker="."); ax.set_ylim(0, 101); ax.set_title("arithmetic check pass rate by filing year (sample, %)")
    plt.show()
    print("Interpretation: checks fail legitimately on filings that present rounded or partial statements; a rate far below the")
    print("population's usual (~90-97%) for a recent year points at a builder regression for that quarter's FSDS layout.")
    score("statement arithmetic checks pass rate (sample)", f"{overall:.1f}%", ">= 90%", "PASS" if overall >= 90 else ("WARN" if overall >= 80 else "FAIL"))

In [ ]:
if FULL_SCANS:
    built = q("SELECT fsds_quarter AS quarter, count(DISTINCT accession) AS filings_with_statements FROM statements WHERE source = 'fsds' GROUP BY 1 ORDER BY 1")
    subs = q("SELECT quarter, count(DISTINCT adsh) AS sub_filings FROM fsds_sub GROUP BY 1 ORDER BY 1")
    cov = subs.merge(built, on="quarter", how="left").fillna(0); cov["built_pct"] = 100 * cov.filings_with_statements / cov.sub_filings
    fig, ax = plt.subplots(figsize=(12, 4)); ax.plot(cov.quarter, cov.built_pct, marker="."); ax.set_ylim(0, 105); ax.set_title("filings in the SEC quarterly file that got statements (%)"); ax.set_xticks(range(0, len(cov), 4)); ax.set_xticklabels(cov.quarter[::4], rotation=45); plt.show()
    display(cov.tail(8))
    low = cov[cov.built_pct < 80]
    score("FSDS quarters with under 80% of filings built", low.quarter.tolist() or "none", "none", "PASS" if len(low) == 0 else "WARN")
    total_c = q("SELECT count(DISTINCT cik) AS c FROM statements").c.iloc[0]
    with_f = q("SELECT count(DISTINCT cik) AS c FROM filings WHERE form IN ('10-K','10-Q','20-F','40-F') AND year >= 2009").c.iloc[0]
    print(f"companies with statements: {total_c:,}; companies with a 10-K/10-Q/20-F/40-F since 2009: {with_f:,} ({pct(total_c, with_f):.1f}%)")
    score("companies with statements among periodic filers since 2009", f"{pct(total_c, with_f):.1f}%", ">= 70%", "PASS" if pct(total_c, with_f) >= 70 else "WARN", "smaller filers often lack FSDS presentations before 2011")
else:
    ll_sub = ll[ll.table == "sub"].set_index("quarter").loaded_rows if len(ll) else pd.Series(dtype=float)
    print("Whole-table quarter coverage needs a local lake; the FSDS load log above shows the quarterly file sizes that were loaded.")
    print("Companies with statements (population) is reported by /coverage on a local lake.")

## 6. Periods and headline metrics

**Checked:** the periods table (one row per fiscal period per company, pointing at the filing that
reported it) and the company metrics (revenue, net income, EPS, assets, operating cash flow per
period) for integrity: every results accession must exist in filings, fiscal labels must be
consistent, metrics must cover the periods they can. **Meaning:** these small tables drive the
company page's headline numbers and the period list; a dangling accession is a page that cannot
open its filing. **Bad looks like:** dangling accessions, companies with periods but no metrics
despite having statements, duplicate period labels.

In [ ]:
periods = q("SELECT cik, period_label, fiscal_year, fiscal_quarter, period_type, period_end, results_accession, results_form, results_filed_date, earnings_release_accession, label_method FROM periods")
metrics = q("SELECT * FROM company_metrics")
print(f"periods: {len(periods):,} for {periods.cik.nunique():,} companies; metrics: {len(metrics):,} for {metrics.cik.nunique():,} companies")
dup_labels = int(periods.duplicated(["cik", "period_label"]).sum())
lag = (pd.to_datetime(periods.results_filed_date) - pd.to_datetime(periods.period_end)).dt.days
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
periods.period_type.value_counts().plot.bar(ax=axes[0], color="#264653"); axes[0].set_title("period types"); axes[0].set_xlabel("")
axes[1].hist(lag.clip(0, 400).dropna(), bins=40, color="#2a9d8f"); axes[1].set_title("days from period end to results filing"); axes[1].axvline(90, color="k", ls="--", lw=0.8)
periods.label_method.fillna("(none)").value_counts().plot.bar(ax=axes[2], color="#e9c46a"); axes[2].set_title("how the fiscal label was derived"); axes[2].set_xlabel("")
plt.tight_layout(); plt.show()
print(f"median filing lag: {lag.median():.0f} days (10-Q due 40-45 days, 10-K 60-90 days after period end)")
score("periods: duplicate (cik, period_label)", dup_labels, 0, "PASS" if dup_labels == 0 else "FAIL")
negative = int((lag < 0).sum())
score("periods: results filed before the period ended", negative, "0 (a few are SEC data errors)", "PASS" if negative == 0 else "WARN")
# dangling accessions: on a local lake check all; on a remote one check the sample
if FULL_SCANS:
    dangling = q("SELECT count(*) AS n FROM periods p LEFT JOIN filings f ON f.accession = p.results_accession WHERE f.accession IS NULL").n.iloc[0]
    scope = "all periods"
else:
    ps = periods[periods.cik.isin(SAMPLE)]
    have = set(pd.concat([q("SELECT accession FROM filings WHERE cik = ?", [c]) for c in SAMPLE]).accession)
    dangling = int((~ps.results_accession.isin(have)).sum()); scope = "sample"
score("periods: results accession missing from filings", int(dangling), 0, "PASS" if dangling == 0 else "FAIL", scope)

In [ ]:
# company_metrics holds ONE row per company: its latest annual numbers (for peer lists and ranking).
per_company = metrics.groupby("cik").size()
print(f"metrics rows per company: min {per_company.min()}, max {per_company.max()} (one row each is the design)")
latest_fy = metrics.fiscal_year.value_counts().sort_index()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(latest_fy.index, latest_fy.values, color="#264653"); axes[0].set_title("companies by the fiscal year of their latest annual numbers")
recent = metrics[metrics.fiscal_year >= 2009]
filled_recent = recent[["revenue", "net_income", "eps_diluted", "total_assets", "operating_cash_flow"]].notna().mean() * 100
filled_all = metrics[["revenue", "net_income", "eps_diluted", "total_assets", "operating_cash_flow"]].notna().mean() * 100
pd.DataFrame({"latest FY >= 2009": filled_recent, "all companies": filled_all}).plot.bar(ax=axes[1]); axes[1].set_ylim(0, 100); axes[1].set_title("share of metric rows with each headline number (%)"); axes[1].set_xlabel("")
plt.tight_layout(); plt.show()
stale = int((metrics.fiscal_year < 2009).sum())
have_stmts = set(stmts._cik.unique()) if len(stmts) else set()
with_metrics = pct(len(have_stmts & set(metrics.cik)), len(have_stmts)) if have_stmts else float("nan")
print(f"companies whose latest annual period predates 2009 (no FSDS statements, so their numbers are blank): {stale:,} of {len(metrics):,}")
print(f"sampled companies with statements that have a metrics row: {with_metrics:.0f}%")
print("Interpretation: a metrics row with blanks is a company whose newest annual filing is older than the statement data sets")
print("(defunct or dormant filers); the fill rate among companies with a recent annual filing is what the product relies on.")
print("Revenue is absent for banks and holding companies by presentation, EPS for entities without shares.")
score("metrics: total_assets filled (latest FY >= 2009)", f"{filled_recent.total_assets:.1f}%", ">= 90%", "PASS" if filled_recent.total_assets >= 90 else "WARN")
score("metrics: net_income filled (latest FY >= 2009)", f"{filled_recent.net_income:.1f}%", ">= 90%", "PASS" if filled_recent.net_income >= 90 else "WARN")
score("metrics: one row per company", f"{per_company.max()} max", "1", "PASS" if per_company.max() == 1 else "FAIL")
score("sampled companies with statements that have a metrics row", f"{with_metrics:.0f}%", ">= 95%", "PASS" if with_metrics >= 95 else "WARN")

## 7. Reconciliation: statements (FSDS) against facts (companyfacts)

**Checked:** for the sampled companies, the headline lines the product shows (revenue, net income,
total assets, operating cash flow) taken from the statements table are compared with the same
concept in the facts table for the same filing and period end. The two tables come from **different
SEC publications** (the quarterly financial statement data sets versus the per-company XBRL API),
so agreement is strong evidence that both were loaded correctly and that the statement builder
kept values intact. **Meaning:** a match rate near 100% with tiny relative differences (rounding
of presented values) is the expected picture. **Bad looks like:** systematic differences for one
concept (a scale or sign error) or one statement type.

In [ ]:
CONCEPTS = {
    "revenue": ["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax", "SalesRevenueNet"],
    "net_income": ["NetIncomeLoss", "ProfitLoss"],
    "total_assets": ["Assets"],
    "operating_cash_flow": ["NetCashProvidedByUsedInOperatingActivities"],
}
rec_rows = []
if len(stmts) and len(facts_s):
    JOIN = ["_cik", "accession", "concept", "period_start", "period_end"]   # a 10-K carries quarterly and annual values ending the same day
    st = stmts[stmts.concept.isin(sum(CONCEPTS.values(), []))][JOIN + ["value"]].dropna(subset=["value"])
    fa = facts_s[facts_s.concept.isin(sum(CONCEPTS.values(), []))][JOIN + ["value"]].dropna(subset=["value"])
    for d in (st, fa):
        d["period_start"] = pd.to_datetime(d.period_start).dt.date; d["period_end"] = pd.to_datetime(d.period_end).dt.date
    fa = fa.drop_duplicates(JOIN); st = st.drop_duplicates(JOIN)
    j = st.merge(fa, on=JOIN, how="inner", suffixes=("_stmt", "_fact"))
    j["rel_diff"] = (j.value_stmt - j.value_fact).abs() / j.value_fact.abs().replace(0, np.nan)
    j["rel_diff"] = j.rel_diff.fillna((j.value_stmt - j.value_fact).abs())
    j["metric"] = j.concept.map({c: k for k, cs in CONCEPTS.items() for c in cs})
    print(f"matched statement lines with a fact: {len(j):,} (of {len(st):,} headline statement lines in the sample)")
    summary = j.groupby("metric").agg(pairs=("rel_diff", "size"), exact=("rel_diff", lambda r: int((r <= 1e-6).sum())), within_1pct=("rel_diff", lambda r: int((r <= 0.01).sum())), median_rel_diff=("rel_diff", "median"))
    summary["exact_pct"] = 100 * summary.exact / summary.pairs; summary["within_1pct_pct"] = 100 * summary.within_1pct / summary.pairs
    display(summary)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    for metric, g in j.groupby("metric"):
        axes[0].scatter(np.sign(g.value_fact) * np.log10(g.value_fact.abs() + 1), np.sign(g.value_stmt) * np.log10(g.value_stmt.abs() + 1), s=6, alpha=0.5, label=metric)
    axes[0].plot([-12, 12], [-12, 12], "k--", lw=0.7); axes[0].set_xlabel("facts value (signed log10)"); axes[0].set_ylabel("statements value (signed log10)"); axes[0].legend(fontsize=8); axes[0].set_title("statements vs facts, headline concepts")
    axes[1].hist(np.log10(j.rel_diff[j.rel_diff > 0] + 1e-12), bins=30, color="#e76f51"); axes[1].set_title(f"log10 relative difference where not exact ({int((j.rel_diff > 1e-6).sum())} of {len(j):,})")
    plt.tight_layout(); plt.show()
    worst = j.sort_values("rel_diff", ascending=False).head(10)
    worst["name"] = worst._cik.map(names).str.slice(0, 30); display(worst[["name", "accession", "metric", "period_end", "value_stmt", "value_fact", "rel_diff"]])
    agree = pct(int((j.rel_diff <= 0.01).sum()), len(j))
    print("Interpretation: FSDS values are the reported figures; companyfacts holds the same XBRL facts, so exact agreement is")
    print("expected. Differences within 1% come from filings that restate a comparative; larger ones are loader or scale errors.")
    score("statements vs facts: headline values within 1% (sample)", f"{agree:.2f}%", ">= 98%", "PASS" if agree >= 98 else ("WARN" if agree >= 95 else "FAIL"))
    rec_rows = j
else:
    print("needs sampled statements and facts")

## 8. Timeliness and drift

**Checked:** how current the lake is: the latest filing date it holds against today, and in `api`
mode the newest filing the SEC lists for each sampled company against the newest in the lake.
**Meaning:** the daily refresh should keep the gap at a business day or two. **Bad looks like:** a
gap of weeks (the refresh is not running) or sampled companies whose newest SEC filing is missing.

In [ ]:
latest = q(f"SELECT max(filed_date) AS d FROM filings WHERE year >= {TODAY.year}").d.iloc[0]
latest = pd.to_datetime(latest).date() if latest is not None else None
gap = (TODAY - latest).days if latest else None
print(f"latest filing in the lake: {latest}   gap to today: {gap} days")
score("lake currency: days since the newest filing", gap, "<= 3 business days", "PASS" if gap is not None and gap <= 5 else "WARN")
if SOURCE_MODE == "api" and filing_cmp:
    drift = pd.DataFrame(filing_cmp)[["name", "missing", "missing_before_lake_max_date"]]
    drift["new_since_snapshot"] = drift.missing - drift.missing_before_lake_max_date
    display(drift.sort_values("new_since_snapshot", ascending=False).head(10))
    print("Interpretation: 'new_since_snapshot' are filings the SEC published after the lake's newest filing; the refresh will add them.")

## 9. Scorecard

Every check above with its value, the threshold it was judged against and the outcome. `PASS` needs no
action. `WARN` is worth a look: it is either an expected limitation stated in the section, or a sign of
drift. `FAIL` is data the product would show wrongly or not at all, and should be fixed before the
next refresh (the section names the table and the loader involved).

In [ ]:
sc = pd.DataFrame(SCORECARD)
order = {"FAIL": 0, "WARN": 1, "PASS": 2}
sc = sc.sort_values("status", key=lambda s: s.map(order))
def _color(s):
    return ["background-color: #f8d7da" if v == "FAIL" else "background-color: #fff3cd" if v == "WARN" else "background-color: #d4edda" for v in s]
display(sc.style.apply(_color, subset=["status"]))
counts = sc.status.value_counts()
print(f"{counts.get('PASS', 0)} pass, {counts.get('WARN', 0)} warn, {counts.get('FAIL', 0)} fail   |   lake: {storage.root}   source mode: {SOURCE_MODE}   sample: {len(SAMPLE)} companies")

## How to read this audit

- **Loader correctness** is answered by sections 3 and 4 in `raw` or `api` mode (recall and value
  agreement against the SEC's own records) and by section 7 in every mode (two independent SEC
  publications agreeing with each other after two independent loaders).
- **Coverage** is answered by sections 1, 5 and 6: which quarters, which companies, which statement
  types, and whether the derived tables (periods, metrics) were rebuilt after the statements.
- **Currency** is section 8: run the daily refresh workflow with the lake secrets set and the gap
  closes on its own.
- Known, accepted limitations: statements exist from 2009q2 (the SEC's data sets start there);
  companies without XBRL presentations get fallback statements built from facts; whole-lake figures
  need a local lake.